# Lab: Kafka and Spark Structured Streaming, in Google Colab version 2

## What you are going to build

By the end you will have a small but complete streaming pipeline running inside one Colab
machine:

```
  your Python code                Kafka broker                 Spark
  ----------------                ------------                 -----
  producer.send()  ----------->   topic "clicks"   ---------->  readStream
  (Step 3)                        3 partitions                  (Step 5)
                                  (Step 2)                          |
                                                                    v
                                                            counts per user,
                                                            updated every 2s
```

Three programs talking to each other. You write the first and the third, Kafka is the
middle one, and the whole point is that the first and third never speak directly.

## A vocabulary primer, before any code

Read this once. Everything below uses these words, and they are the only genuinely new
ideas in the lab.

### Batch versus stream

This is the central distinction of the whole course.

| | Batch | Stream |
|---|---|---|
| The data | A file that has an end | A feed that has no end |
| The question | "How many clicks yesterday?" | "How many clicks so far, updated constantly?" |
| You run it | Once, then it finishes | Once, then it keeps running |
| Analogy | A photograph | A video |

A **batch** job reads data that already finished arriving. A **stream** job reads data that
is still arriving, and keeps its answer up to date as more shows up.

The surprising thing you will see in Step 5 is how little the code changes between the two.

### The four pieces of Kafka

| Word | What it actually is |
|---|---|
| **Broker** | The server that stores messages and hands them out. Kafka *is* the broker. |
| **Topic** | A named list of messages that you add to the end of. Like a log book. |
| **Producer** | Any program that writes messages into a topic. You write one in Step 3. |
| **Consumer** | Any program that reads messages out of a topic. Spark is yours, in Step 4. |

A topic is **append only**: you add to the end, and you never edit or delete what is
already there. A consumer reading a topic does not remove anything, so ten different
consumers can read the same topic without interfering with each other. That is the single
most useful property Kafka has.

### Partitions and offsets

A topic is split into **partitions**, which are the slices Kafka can put on different
machines. Our topic will have 3.

Every message in a partition gets an **offset**, a number counting up from 0. Offsets are
per partition, not per topic. So "message 5" is meaningless unless you also say which
partition.

Picture the topic as three separate numbered lists rather than one:

```
  partition 0:  [0] [1] [2] [3] [4] ...
  partition 1:  [0] [1] ...
  partition 2:  [0] [1] [2] ...
```

### Keys

When you send a message you may attach a **key**. Kafka hashes the key to pick the
partition, so **the same key always goes to the same partition**. We will use the user name
as the key, which means every event from alice lands in one partition. Step 4 proves it.

### Why bother with a broker at all?

You could have your producer talk straight to Spark. Companies do not, for one reason:
**decoupling**. With a broker in the middle, the producer does not need to know who is
reading, how many readers there are, or whether they are even running right now. Add a
second consumer next year and the producer never changes. Take the consumer down for
maintenance and the messages wait in the topic.

**Check yourself.** A topic has 3 partitions. A consumer reads a message with offset 7. Is that the 8th message in the topic?

<details>
<summary>Click for the answer</summary>

No. It is the 8th message **in its partition**. Offsets count up inside one partition only, so the topic could easily have 30 messages by then. There is no global ordering across a topic, and that is the price Kafka pays for being able to spread one topic across many machines.

</details>

## Step 0. Install everything

### One idea you need first: the JVM

Both Kafka and Spark are written in **Java**, not Python. They run on the **JVM**, the Java
Virtual Machine, which is the program that executes Java code.

So when you write `spark.read...` in Python, your Python is not doing the work. It is
sending instructions to a JVM running alongside it, and the JVM does the real work. This is
why the first cell installs Java even though you will write no Java at all, and it explains
a lot of otherwise strange behavior later, such as why starting a process can take seconds.

### Cell 1 of 4: Java

**What it does.** Checks whether this Colab machine already has Java 17 or newer, and
installs it only if it does not. Spark 4 refuses to run on anything older.

**Line by line.**

| Code | Meaning |
|---|---|
| `import os, subprocess, time, re` | Four standard libraries: `os` for environment variables, `subprocess` for running other programs, `time` for timing, `re` for pattern matching. |
| `def java_major(home=None)` | A small helper that answers "which major version of Java is this?" Returns `0` if Java is missing. |
| `subprocess.run([exe, '-version'], ...)` | Runs `java -version` and captures its output instead of printing it. |
| `.stderr` | Not a typo. `java -version` prints its version to the **error** stream, not the normal one. A real historical quirk. |
| `re.search(r'version "(\d+)', out)` | Finds the first number inside `version "17.0.20"`. The `\d+` means "one or more digits" and the parentheses capture it. |
| `except FileNotFoundError: return 0` | If there is no `java` command at all, do not crash: report version 0 so the code below installs it. |
| `os.environ['JAVA_HOME'] = ...` | An **environment variable**: a setting every program started from here can read. Spark looks for `JAVA_HOME` to find Java. |
| `os.environ['PATH'] = JAVA_HOME + '/bin:' + ...` | `PATH` is the list of folders the shell searches for commands. Putting Java's folder first makes `java` mean *our* Java. |
| `!apt-get install -y openjdk-17-jre-headless` | The `!` means "run this as a terminal command, not Python". `apt-get` is Ubuntu's installer. |

**Why `jre-headless` and not `jdk`?** A **JDK** is the full kit including the compiler, for
people writing Java. A **JRE** only runs Java programs. We are only running them, so the JRE
is enough and it is a smaller download. `headless` means no graphical interface, which a
server does not need.

In [ ]:
# --- Java ----------------------------------------------------------------
# Spark 4 needs Java 17 or newer. Colab sometimes already has it, so check before
# installing: the apt install is the slowest part of this cell when it is needed.
import os, subprocess, time, re

t0 = time.time()

def java_major(home=None):
    exe = (home + '/bin/java') if home else 'java'
    try:
        out = subprocess.run([exe, '-version'], capture_output=True, text=True).stderr
    except FileNotFoundError:
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0

JAVA_HOME = '/usr/lib/jvm/java-17-openjdk-amd64'

if java_major() >= 17:
    print('Java 17+ already here, nothing to install')
    JAVA_HOME = subprocess.run(
        ['bash', '-lc', 'dirname $(dirname $(readlink -f $(which java)))'],
        capture_output=True, text=True).stdout.strip()
elif java_major(JAVA_HOME) >= 17:
    print('Java 17 already installed at', JAVA_HOME)
else:
    print('installing Java 17 (JRE only, which is all Spark and Kafka need)...')
    # jre-headless, not jdk-headless: no compiler, so a smaller download.
    !apt-get install -y openjdk-17-jre-headless -qq > /dev/null 2>&1

os.environ['JAVA_HOME'] = JAVA_HOME
os.environ['PATH'] = JAVA_HOME + '/bin:' + os.environ['PATH']
print('JAVA_HOME =', JAVA_HOME)
!java -version
print('--- %.1f seconds ---' % (time.time() - t0))

Java 17+ already here, nothing to install
JAVA_HOME = /usr/lib/jvm/java-21-openjdk-amd64
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)
--- 0.5 seconds ---


**What you should see.** Either "Java 17+ already here" or a short install, then a version
banner like `openjdk version "17.0.x"`. Anything 17 or above is fine. The timing line at the
end is there so you can see where Step 0's time actually goes.

### Cell 2 of 4: PySpark and a Kafka client

**What it does.** Installs two Python packages.

**The two packages.**

- **`pyspark`** is Spark itself plus the Python API you will write in. It is a large
  download, around 400 MB, because it contains the whole Java engine. This is the slowest
  cell in Step 0 and there is no way around it.
- **`kafka-python`** is a Kafka **producer and consumer you drive from Python**. Step 3 uses
  it to send messages.

**Why `==4.0.1` and not just `pyspark`?** Writing `pyspark==4.0.1` **pins** the version.
Without a pin, pip installs whatever is newest, which could be a version whose Kafka
connector jars do not match the ones Step 0 downloads. Pinning is what makes this notebook
still work next year. In your own projects, pin anything you depend on.

**What the flags mean.** `-q` is quiet, so pip prints a few lines instead of a few hundred.

In [ ]:
# --- PySpark and a Kafka client ------------------------------------------
# kafka-python is a producer you drive from Python. Version 1 sent messages by
# launching kafka-console-producer.sh, which starts a whole JVM for every call:
# about 1 to 2 seconds per message. This sends 8 messages in about 0.15 seconds.
import time
t0 = time.time()

!pip install -q pyspark==4.0.1 kafka-python

import pyspark
print('pyspark     ', pyspark.__version__)
import kafka
print('kafka-python', kafka.__version__)
print('--- %.1f seconds ---' % (time.time() - t0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 614.2/614.2 kB 21.2 MB/s eta 0:00:00
pyspark      4.0.1
kafka-python 3.0.11
--- 76.4 seconds ---


**What you should see.** Two version numbers, `pyspark 4.0.1` and a `kafka-python` version.
If pip prints a red warning about dependency conflicts you can usually ignore it: Colab
pre-installs many packages and they do not all agree with each other.

### Cell 3 of 4: Kafka itself

**What it does.** Downloads Kafka and unpacks it.
**Line by line.**

| Code | Meaning |
|---|---|
| `urllib.request.urlopen(...).read().decode()` | Fetch a web page and turn the raw bytes into text. |
| `re.findall(r'\d+\.\d+\.\d+/', html)` | Find every `4.3.1/` style version in the page. `\.` means a literal dot, because a bare `.` in a pattern matches any character. |
| `{v.rstrip('/') for v in ...}` | A **set comprehension**: strip the trailing slash off each, and the curly braces throw away duplicates. |
| `key=lambda s: [int(p) for p in s.split('.')]` | Sort **numerically**, not alphabetically. This matters: as text, `"4.9.0"` sorts *after* `"4.10.0"`, which would pick the wrong version. Turning it into `[4, 9, 0]` and `[4, 10, 0]` fixes that. |
| `try: ... except Exception as e:` | If the CDN cannot be reached for any reason, fall back to the slow archive rather than failing. Slow but working beats fast and broken. |
| `!wget -q --show-progress -O /content/kafka.tgz {URL}` | Download the file. `{URL}` is Colab substituting a Python variable into a shell command. `--show-progress` is deliberate: a silent 10 second wait looks like a freeze. |
| `!tar xzf ... --strip-components=1` | Unpack the archive. A `.tgz` is a folder squashed into one compressed file. `--strip-components=1` drops the pointless top folder so you get `kafka/bin` rather than `kafka/kafka_2.13-4.3.1/bin`. |
| `if not os.path.exists(KAFKA_DIR + '/bin')` | Skip all of this if Kafka is already unpacked, so re-running the cell is free. |

In [ ]:
# --- Kafka ----------------------------------------------------------------
# The CDN at dlcdn.apache.org only carries current releases, and which release is
# current changes over the year. So ask it what it has rather than hard-coding a
# version that will quietly disappear. archive.apache.org has every version ever
# released but is throttled to about 120 KB/s, so it is the fallback, not the default.
import os, re, time, urllib.request

KAFKA_DIR = '/content/kafka'
t0 = time.time()

def newest_on_cdn():
    html = urllib.request.urlopen(
        'https://dlcdn.apache.org/kafka/', timeout=30).read().decode()
    vers = sorted({v.rstrip('/') for v in re.findall(r'\d+\.\d+\.\d+/', html)},
                  key=lambda s: [int(p) for p in s.split('.')])
    return vers[-1] if vers else None

if not os.path.exists(KAFKA_DIR + '/bin'):
    try:
        KAFKA_VERSION = newest_on_cdn()
        URL = 'https://dlcdn.apache.org/kafka/%s/kafka_2.13-%s.tgz' % (
            KAFKA_VERSION, KAFKA_VERSION)
        print('CDN has Kafka %s, downloading at CDN speed...' % KAFKA_VERSION)
    except Exception as e:
        KAFKA_VERSION = '3.9.1'
        URL = ('https://archive.apache.org/dist/kafka/%s/kafka_2.13-%s.tgz'
               % (KAFKA_VERSION, KAFKA_VERSION))
        print('CDN unreachable (%s). Falling back to the slow archive.' % e)

    !wget -q --show-progress -O /content/kafka.tgz {URL}
    !mkdir -p {KAFKA_DIR}
    !tar xzf /content/kafka.tgz -C {KAFKA_DIR} --strip-components=1
    !rm -f /content/kafka.tgz
else:
    print('Kafka already unpacked')

!{KAFKA_DIR}/bin/kafka-topics.sh --version
print('--- %.1f seconds ---' % (time.time() - t0))

CDN has Kafka 4.4.0, downloading at CDN speed...
/content/kafka.tgz  100%[===================>] 129.03M   402MB/s    in 0.3s    
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
4.4.0
--- 8.6 seconds ---


**What you should see.** A progress bar, then a version number such as `4.3.1`, then a
timing line of roughly 6 to 15 seconds.

### Cell 4 of 4: the Spark Kafka connector

**What it does.** Downloads four `.jar` files into Spark's own library folder.

**What is a jar?** A **jar** is a zip file full of compiled Java code: Java's equivalent of
a Python package. Spark loads jars at startup to learn new abilities.

**Why Spark needs these.** Spark does not know how to talk to Kafka out of the box. That
ability lives in a separate connector. Without these four files you get the error
`Failed to find data source: kafka`, which is one of the most common Spark errors there is,
and now you know exactly what it means: a missing jar.

**Why download them by hand.** The normal approach is one line:

```python
.config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1')
```

That tells Spark to fetch the connector from **Maven**, the main public repository of Java
libraries, using a resolver called **Ivy**. Ivy works out the full dependency tree, which
takes minutes in Colab and sometimes stalls completely. Since we already know exactly which
four files are needed, downloading them directly takes about a second and the SparkSession
then starts with no network access at all.

**The four files, and why each is there.**

| Jar | Job |
|---|---|
| `spark-sql-kafka-0-10` | The connector: teaches Spark that `format('kafka')` is a thing. |
| `spark-token-provider-kafka-0-10` | Authentication support. Needed even though our broker has no security, because the connector refers to it. |
| `kafka-clients` | Kafka's own official client library. The connector is a wrapper around this. |
| `commons-pool2` | A connection pool, so Spark reuses Kafka connections instead of opening a new one per task. |

**Line by line.**

| Code | Meaning |
|---|---|
| `os.path.dirname(pyspark.__file__)` | Every Python module knows its own file location. This finds where pip put PySpark, whose `jars` folder is where Spark looks. |
| `SV = pyspark.__version__` | Build the URLs from the installed version, so the connector always matches Spark. A mismatch here is a classic source of confusing errors. |
| `kafka-clients/3.9.1` and `commons-pool2/2.12.0` | These two are **hard-coded**, because they come from the connector's own dependency list rather than from Spark's version number. |
| `if not os.path.exists(dest)` | Do not re-download on a re-run. |

In [ ]:
# --- The Spark Kafka connector -------------------------------------------
# Spark cannot read Kafka without four extra jars. The usual way is
# .config('spark.jars.packages', ...), which makes Spark run Ivy to resolve them
# from Maven on every start: often a few minutes in Colab, and it sometimes stalls.
# Downloading the four files straight into Spark's own jars folder takes about a
# second, and then the SparkSession starts with no network calls at all.
import os, time, pyspark
t0 = time.time()

JARS = os.path.join(os.path.dirname(pyspark.__file__), 'jars')
MAVEN = 'https://repo1.maven.org/maven2'
SV = pyspark.__version__

# kafka-clients and commons-pool2 versions come from the connector's own pom for
# Spark 4.0.1. If you change the PySpark version, check that pom again.
WANTED = [
    ('%s/org/apache/spark/spark-sql-kafka-0-10_2.13/%s/'
     'spark-sql-kafka-0-10_2.13-%s.jar' % (MAVEN, SV, SV),
     'spark-sql-kafka-0-10_2.13-%s.jar' % SV),
    ('%s/org/apache/spark/spark-token-provider-kafka-0-10_2.13/%s/'
     'spark-token-provider-kafka-0-10_2.13-%s.jar' % (MAVEN, SV, SV),
     'spark-token-provider-kafka-0-10_2.13-%s.jar' % SV),
    ('%s/org/apache/kafka/kafka-clients/3.9.1/kafka-clients-3.9.1.jar' % MAVEN,
     'kafka-clients-3.9.1.jar'),
    ('%s/org/apache/commons/commons-pool2/2.12.0/commons-pool2-2.12.0.jar' % MAVEN,
     'commons-pool2-2.12.0.jar'),
]

for url, name in WANTED:
    dest = os.path.join(JARS, name)
    if not os.path.exists(dest):
        !wget -q -O {dest} {url}
    print('%7.1f KB  %s' % (os.path.getsize(dest) / 1024, name))

print('--- %.1f seconds ---' % (time.time() - t0))

  456.6 KB  spark-sql-kafka-0-10_2.13-4.0.1.jar
   63.5 KB  spark-token-provider-kafka-0-10_2.13-4.0.1.jar
 8999.2 KB  kafka-clients-3.9.1.jar
  146.5 KB  commons-pool2-2.12.0.jar
--- 0.8 seconds ---


**What you should see.** Four lines with file sizes, the largest being `kafka-clients` at
about 9 MB, and a timing line of a second or two.

### Recap of Step 0

| Piece | What it is |
|---|---|
| **Java 17** | The engine Kafka and Spark both run on. Python only drives them. |
| **PySpark 4.0.1** | Spark, plus its Python API. |
| **kafka-python** | A producer you can call from Python. |
| **Kafka** | The broker: the thing that stores and serves messages. |
| **Four jars** | The bridge that lets Spark read a Kafka topic as a DataFrame. |

Notice what is **not** here: **ZooKeeper**. Older Kafka needed a second cluster, ZooKeeper,
just to keep track of which broker was in charge. Modern Kafka does that itself in **KRaft
mode**.

## Step 1. Start the broker

Two things must happen before a broker will run: its storage has to be **formatted**, and
then the server process has to **start**.

### Why "format" is not as alarming as it sounds

Formatting here means writing a small file called `meta.properties` into an empty folder,
recording a **cluster ID**. It is closer to writing your name in a new notebook than to
wiping a hard drive. Kafka insists on it because a broker must know which cluster it belongs
to before it will accept data.

### Why this cell has an if/else about config paths

Step 0 downloads whatever version the CDN currently serves, and Kafka changed two things
between version 3 and version 4:

| | Kafka 3.x | Kafka 4.x |
|---|---|---|
| Config file | `config/kraft/server.properties` | `config/server.properties` |
| Format command | works as is | **requires** `--standalone` |

Rather than guess, the cell looks to see which layout it actually got. This is a good habit
in general: detect, do not assume.

**Line by line.**

| Code | Meaning |
|---|---|
| `os.path.exists(...)` | Asks the filesystem which Kafka version's layout we have. |
| `FORMAT_EXTRA = ['--standalone']` | An extra command line flag, collected in a list so it can be added or left out cleanly. In Kafka 4, `--standalone` means "this one machine is the whole cluster". |
| `def kafka(*args)` | A helper so the rest of the notebook can write `kafka('kafka-topics.sh', '--list')` instead of a long path every time. The `*args` means "accept any number of arguments". |
| `subprocess.run(cmd, capture_output=True, text=True).stdout` | Run a program, capture what it prints, and give it back as a string rather than raw bytes. |
| `kafka-storage.sh random-uuid` | Asks Kafka to invent a **UUID**, a long random identifier, to be this cluster's ID. |
| `!sed -i "s#^log.dirs=.*#log.dirs={LOG_DIR}#" {CONFIG}` | Edits the config file in place. `sed` is a find-and-replace tool; this one rewrites the line starting with `log.dirs=` so Kafka stores data where we want. The `#` characters are just the separator, used instead of `/` because the replacement text contains slashes. |
| `--ignore-formatted` | "If this is already formatted, do not complain." Makes re-running the cell safe. |

In [ ]:
import subprocess, time, os, glob

LOG_DIR = '/content/kraft-logs'

# Where the config lives moved between Kafka 3 and Kafka 4, and so did the format
# command. Detect which layout we got, so this cell keeps working whichever version
# the CDN happens to be serving.
if os.path.exists(KAFKA_DIR + '/config/kraft/server.properties'):
    CONFIG = KAFKA_DIR + '/config/kraft/server.properties'   # Kafka 3.x
    FORMAT_EXTRA = []
else:
    CONFIG = KAFKA_DIR + '/config/server.properties'         # Kafka 4.x
    FORMAT_EXTRA = ['--standalone']   # 4.x refuses to format without this

print('using config:', CONFIG)

def kafka(*args):
    """Run a Kafka command line tool and return its output."""
    cmd = [KAFKA_DIR + '/bin/' + args[0]] + list(args[1:])
    return subprocess.run(cmd, capture_output=True, text=True).stdout

if not os.path.exists(LOG_DIR + '/meta.properties'):
    cluster_id = subprocess.run([KAFKA_DIR + '/bin/kafka-storage.sh', 'random-uuid'],
                                capture_output=True, text=True).stdout.strip()
    print('cluster id:', cluster_id)
    !sed -i "s#^log.dirs=.*#log.dirs={LOG_DIR}#" {CONFIG}
    subprocess.run([KAFKA_DIR + '/bin/kafka-storage.sh', 'format',
                    '-t', cluster_id, '-c', CONFIG,
                    '--ignore-formatted'] + FORMAT_EXTRA,
                   capture_output=True, text=True)
    print('storage formatted at', LOG_DIR)
else:
    print('storage already formatted')

using config: /content/kafka/config/server.properties
cluster id: [0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
MU1UfttwTpmpW6itEMLjDw
storage formatted at /content/kraft-logs


**What you should see.** The config path it chose, a cluster ID such as
`wTiYHroURca9hfyPBMS3pA`, and "storage formatted". On a second run it says "storage already
formatted", which is correct and not a problem.

### Starting the server

**The new idea: a background process.** Everything you have run so far finished and returned
control to you. A server does not finish: it runs until stopped. If we started it the normal
way, the cell would hang forever and you could never run the next one.

So we start it **in the background**, which means Python launches it and immediately carries
on, leaving it running alongside.

**Line by line.**

| Code | Meaning |
|---|---|
| `subprocess.Popen([...])` | Start a program and **do not wait** for it. The contrast with `subprocess.run`, which does wait, is the whole point. |
| `stdout=open('/content/broker.log', 'w')` | Send everything the broker prints to a file instead of into your notebook, which would otherwise flood with hundreds of log lines. |
| `stderr=subprocess.STDOUT` | Send errors to that same file. |
| `BOOTSTRAP = 'localhost:9092'` | The broker's address. `localhost` means this machine; `9092` is the **port**, Kafka's conventional one. A real cluster lists several machines here, and nothing else in this notebook would change. |
| `for attempt in range(90):` | Try up to 90 times. The broker needs a few seconds to be ready, and asking too early fails. |
| `if 'Error' not in kafka(...)` | A crude but effective readiness check: ask the broker to list topics, and if the answer has no error in it, it is awake. |
| `time.sleep(1)` | Wait a second between attempts rather than hammering it. |
| `else:` on the `for` loop | A genuinely unusual Python feature: a `for` loop's `else` runs **only if the loop finished without `break`**. So this prints the failure message only if all 90 attempts failed. |

**The pattern to notice.** "Start a thing, then poll until it is ready" is how you deal with
every server you will ever start, in every language. Fixing it with `time.sleep(30)` instead
is both slower and less reliable.

In [ ]:
# Start the broker in the background and wait until it answers.
t0 = time.time()
broker = subprocess.Popen(
    [KAFKA_DIR + '/bin/kafka-server-start.sh', CONFIG],
    stdout=open('/content/broker.log', 'w'), stderr=subprocess.STDOUT)

BOOTSTRAP = 'localhost:9092'

for attempt in range(90):
    if 'Error' not in kafka('kafka-topics.sh', '--bootstrap-server', BOOTSTRAP, '--list'):
        print('broker is up after about %.0f seconds' % (time.time() - t0))
        break
    time.sleep(1)
else:
    print('broker did not start. Read the log with:  !tail -40 /content/broker.log')

broker is up after about 10 seconds


**What you should see.** `broker is up after about 2 seconds`. If instead you get "broker did
not start", run `!tail -40 /content/broker.log` in a new cell to see why.

You are now running a real Kafka cluster of one machine. Not a simulator or a mock: the same
software a bank would run, with one broker instead of thirty.

## Step 2. Create a topic

Now make the named log we will write into.

**The options explained.**

| Option | Meaning |
|---|---|
| `--topic clicks` | The name. Consumers subscribe by name. |
| `--partitions 3` | Split it into 3 slices. This is what lets Kafka spread a topic over machines and lets Spark read it in parallel. |
| `--replication-factor 1` | Keep **one** copy of each slice. |

**On replication factor 1.** Replication is how Kafka survives a dead machine: with a factor
of 3, each slice exists on three brokers, so any one can die harmlessly. We use 1 because we
only have one broker, which means **if this machine dies the data is gone**. That is fine for
a throwaway Colab machine and would be unacceptable in production, where 3 is standard.

In [ ]:
print(kafka('kafka-topics.sh', '--bootstrap-server', BOOTSTRAP,
            '--create', '--topic', 'clicks',
            '--partitions', '3', '--replication-factor', '1'))

print(kafka('kafka-topics.sh', '--bootstrap-server', BOOTSTRAP,
            '--describe', '--topic', 'clicks'))

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Created topic clicks.

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Topic: clicks	TopicId: QciMqFZ8Rp6yBPojmD5Wyg	PartitionCount: 3	ReplicationFactor: 1	Configs: min.insync.replicas=1,segment.bytes=1073741824
	Topic: clicks	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
	Topic: clicks	Partition: 1	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 
	Topic: clicks	Partiti

**What you should see, and how to read it.**

```
Topic: clicks   PartitionCount: 3   ReplicationFactor: 1
    Topic: clicks   Partition: 0   Leader: 1   Replicas: 1   Isr: 1
    Topic: clicks   Partition: 1   Leader: 1   Replicas: 1   Isr: 1
    Topic: clicks   Partition: 2   Leader: 1   Replicas: 1   Isr: 1
```

| Field | Meaning |
|---|---|
| `Partition: 0/1/2` | The three slices, as promised. |
| `Leader: 1` | Broker number 1 is in charge of this partition. All reads and writes for it go through the leader. We only have one broker, so it leads all three. |
| `Replicas: 1` | The list of brokers holding a copy. Just broker 1. |
| `Isr: 1` | **In Sync Replicas**: the copies that are actually up to date. If a replica falls behind it drops out of this list, which is the first thing you check when a real cluster is unhealthy. |
| `Elr:` | **Eligible Leader Replicas**, explained below. Empty here, and that is correct. |
| `LastKnownElr:` | Whatever `Elr` held before it last changed. Also empty here. |

### What `Elr` and `LastKnownElr` are

These arrived
with Kafka 4 as a feature called **Eligible Leader Replicas**. Step 1's format output
mentioned `eligible.leader.replicas.version`, which is this feature being switched on.

**The problem they solve.** Kafka refuses to advance the high watermark, the point up to
which data counts as committed, while the ISR is smaller than `min.insync.replicas`. So if
every in-sync replica dies, the old choices were both bad: wait and stay offline, or elect
a replica that is behind and silently lose committed data.

**What ELR is.** A list of replicas that have dropped out of the ISR but are still
considered safe to promote to leader. The controller keeps that list so it has somewhere
safe to turn. Leader election now goes in this order:

1. Pick a member of the **ISR**, if there is one.
2. Otherwise pick an unfenced member of the **ELR**.
3. Otherwise fall back to the last known leader, if it is unfenced.

`LastKnownElr` records what the ELR held before its last change, which the controller uses
while it is working out how to recover.

**Why both are empty here.** Our topic was created with `--replication-factor 1`. There is
exactly one copy of each partition and no followers at all, so nothing can be out of sync
and there is nothing to track. Create a topic with replication factor 3 on a three broker
cluster and these fields start carrying broker ids.

In other words, an empty `Elr` on this lab is not a problem to fix. It is what a
single-broker cluster should look like.

**Check yourself.** Why can Spark read this topic with three tasks in parallel, but not thirty?

<details>
<summary>Click for the answer</summary>

Because parallelism is capped by the number of partitions. Three partitions means at most three readers can work independently; a fourth would have nothing to read. If you need more parallelism, you create the topic with more partitions. This is the main thing you have to get right when creating a topic, because changing it later is awkward.

</details>

## Step 3. Send some messages

Time to write a **producer**.

### What a message actually is

A Kafka message is just **bytes**. Kafka does not know or care what is in it. That is
deliberate, and it is why Kafka can carry anything.

But our data is a Python dictionary, and a dictionary is not bytes. Turning one into the
other is called **serialization**, and turning it back is **deserialization**. We will
serialize to **JSON**, a text format that looks almost exactly like a Python dict:

```json
{"user": "alice", "page": "/cart", "ts": "2026-10-14T09:35:37"}
```

### Line by line

| Code | Meaning |
|---|---|
| `from kafka import KafkaProducer` | The producer class from the `kafka-python` package. |
| `bootstrap_servers=BOOTSTRAP` | Where to find the broker. "Bootstrap" because the producer contacts this address once, then learns the real layout of the cluster from it. |
| `key_serializer=lambda k: k.encode('utf-8')` | How to turn a key into bytes. A `lambda` is a one line unnamed function; this one is "take `k`, return `k.encode('utf-8')`". `.encode()` turns text into bytes, and **UTF-8** is the standard way of doing that. |
| `value_serializer=lambda v: json.dumps(v).encode('utf-8')` | Two steps: `json.dumps` turns the dict into a JSON **string**, then `.encode` turns that string into **bytes**. |
| `def make_event(user=None)` | Builds one fake click. `user=None` is a default, so you may pass a name or let it pick one at random. |
| `random.choice(PAGES)` | Picks a random page. |
| `datetime.datetime.now().isoformat()` | The current time as text, like `2026-10-14T09:35:37.245114`. **ISO format** sorts correctly as text, which is why it is worth using. |
| `producer.send(topic, key=k, value=v)` | Queue a message. Note the word queue. |
| `producer.flush()` | **Wait until the broker has actually confirmed them.** |

### The one line people forget

`send()` does not send. It adds the message to an internal buffer, and the producer ships
whole batches in the background, because one network call for 100 messages is far cheaper
than 100 calls. `flush()` blocks until that buffer is empty and the broker has acknowledged
everything.

Leave `flush()` out and Step 4 can ask for data that has not arrived yet, and you get a
confusing empty answer with no error. If a streaming job of yours ever seems to be missing
the most recent records, this is the first thing to check.

In [ ]:
import json, random, datetime
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP,
    key_serializer=lambda k: k.encode('utf-8'),
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

USERS = ['alice', 'bob', 'carlos', 'diana']
PAGES = ['/home', '/search', '/product/42', '/cart', '/checkout']

def make_event(user=None):
    u = user or random.choice(USERS)
    return u, {'user': u,
               'page': random.choice(PAGES),
               'ts': datetime.datetime.now().isoformat()}

def send(records, topic='clicks'):
    """Send a list of (key, dict) pairs to Kafka."""
    for k, v in records:
        producer.send(topic, key=k, value=v)
    producer.flush()          # wait until the broker has them
    return len(records)

t0 = time.time()
batch = [make_event(u) for u in ['alice', 'bob', 'alice', 'carlos',
                                 'diana', 'bob', 'alice', 'diana']]
n = send(batch)
print('sent %d events in %.3f seconds' % (n, time.time() - t0))
for k, v in batch[:3]:
    print('  key=%-7s value=%s' % (k, json.dumps(v)))

sent 8 events in 0.273 seconds
  key=alice   value={"user": "alice", "page": "/search", "ts": "2026-10-21T14:42:08.267266"}
  key=bob     value={"user": "bob", "page": "/cart", "ts": "2026-10-21T14:42:08.267285"}
  key=alice   value={"user": "alice", "page": "/checkout", "ts": "2026-10-21T14:42:08.267291"}


**What you should see.** `sent 8 events in 0.1 seconds` or thereabouts, then three sample
events.

**Note that timing.** Version 1 of this lab sent each message by launching
`kafka-console-producer.sh`, which starts an entire JVM per message: one to two seconds
each. Here the producer connects once and stays connected, which is how every real
application works. Starting a process is expensive; keeping a connection is cheap. That one
sentence explains a surprising share of all performance problems in data engineering.

## Step 4. Read them back with Spark, as a batch

Before streaming, read the topic as a plain fixed table. A Kafka topic can be read either
way, and the batch view is much easier to inspect because it holds still.

### What a SparkSession is

A **SparkSession** is your handle on the Spark engine: you create one, and every DataFrame
comes from it. Creating it starts the JVM, which is why it takes a few seconds.

**Line by line.**

| Code | Meaning |
|---|---|
| `from pyspark.sql import SparkSession, functions as F` | `F` is a near universal convention for Spark's function library. You will see `F.col`, `F.window` and so on everywhere. |
| `SparkSession.builder...getOrCreate()` | The **builder pattern**: chain settings, then build. `getOrCreate` returns the existing session if there already is one, so re-running the cell is safe. |
| `.appName('KafkaLab')` | A label that shows up in Spark's monitoring UI. Cosmetic here, useful on a shared cluster. |
| `.config('spark.sql.shuffle.partitions', '4')` | How many pieces to split data into when it has to be reorganized. The default is **200**, which is sensible on a real cluster and silly on one small machine, where it creates 200 nearly empty tasks. Lowering it on small data is one of the easiest Spark speedups there is. |
| `setLogLevel('ERROR')` | Print only errors. Spark is extremely chatty by default. |

**Notice what is missing:** there is no `spark.jars.packages` line, because Step 0 already
put the jars where Spark looks. That is why this starts in seconds rather than minutes.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType

t0 = time.time()
spark = (SparkSession.builder
         .appName('KafkaLab')
         .config('spark.sql.shuffle.partitions', '4')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('spark %s, session ready in %.1f seconds' % (spark.version, time.time() - t0))

spark 4.0.1, session ready in 12.2 seconds


### Reading the topic

**Line by line.**

| Code | Meaning |
|---|---|
| `spark.read` | Batch read. The streaming version is `spark.readStream`, and that single word is the only difference. |
| `.format('kafka')` | Use the Kafka connector. This is the line that fails if the jars are missing. |
| `.option('subscribe', 'clicks')` | Which topic to read. |
| `.option('startingOffsets', 'earliest')` | Start from the **beginning** of the topic. The alternative, `latest`, means "only new messages from now on", which is a common cause of an empty result when testing. |
| `.load()` | Build the DataFrame. |
| `raw.count()` | Actually go and do the work. |

### Lazy evaluation, which trips up everyone

`.load()` does **not** read any data. Nor does any filter or column selection you add after
it. Spark records what you asked for and does nothing until you run an **action**, such as
`count()` or `show()`, which demands a real answer.

This is why a line defining a huge computation returns instantly and then `count()` takes
twenty seconds. Spark is not slow at `count()`; it is doing everything at `count()`. The
upside is that Spark sees your whole plan before running any of it, and can optimize across
the lot.

In [ ]:
raw = (spark.read.format('kafka')
       .option('kafka.bootstrap.servers', BOOTSTRAP)
       .option('subscribe', 'clicks')
       .option('startingOffsets', 'earliest')
       .load())

raw.printSchema()
print('rows:', raw.count())

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

rows: 8


**What you should see.** The seven column schema, then `rows: 8`.

### That schema is worth memorizing

It is identical for every Kafka source you will ever read.

| Column | What it is |
|---|---|
| `key` | Your key, **as raw bytes** |
| `value` | Your message, **as raw bytes** |
| `topic` | Which topic it came from |
| `partition` | Which slice |
| `offset` | Its position within that slice |
| `timestamp` | When the broker received it |
| `timestampType` | Whether that time came from the broker or the producer |

The crucial detail is that `key` and `value` are `binary`. Kafka stores bytes and hands back
bytes; it has no idea you put JSON in there. **Casting them to strings is always your first
move.** That is the next cell.

### Turning bytes into something readable

**Line by line.**

| Code | Meaning |
|---|---|
| `selectExpr(...)` | Select columns using **SQL expressions** written as strings, rather than Python. Handy when the SQL is shorter, which casting is. |
| `CAST(key AS STRING)` | Convert the raw bytes to text. |
| `AS user_key` | Rename the result, because `key` is a reserved-ish word and `user_key` is clearer. |

In [ ]:
clicks = raw.selectExpr("CAST(key AS STRING) AS user_key",
                        "CAST(value AS STRING) AS json_value",
                        "partition", "offset", "timestamp")
clicks.orderBy('partition', 'offset').show(10, truncate=False)

+--------+---------------------------------------------------------------------------+---------+------+-----------------------+
|user_key|json_value                                                                 |partition|offset|timestamp              |
+--------+---------------------------------------------------------------------------+---------+------+-----------------------+
|alice   |{"user": "alice", "page": "/checkout", "ts": "2026-10-14T17:26:08.792804"} |0        |0     |2026-10-14 17:26:08.916|
|bob     |{"user": "bob", "page": "/checkout", "ts": "2026-10-14T17:26:08.792818"}   |0        |1     |2026-10-14 17:26:08.916|
|alice   |{"user": "alice", "page": "/home", "ts": "2026-10-14T17:26:08.792821"}     |0        |2     |2026-10-14 17:26:08.916|
|bob     |{"user": "bob", "page": "/product/42", "ts": "2026-10-14T17:26:08.792830"} |0        |3     |2026-10-14 17:26:08.916|
|alice   |{"user": "alice", "page": "/checkout", "ts": "2026-10-14T17:26:08.792833"} |0        |4     |2

**Read the `partition` and `offset` columns together.** You should see something like
partition 0 holding offsets 0 to 4, partition 1 holding offset 0, and partition 2 holding
offsets 0 and 1.

Offsets restart from 0 in every partition. There is no global ordering in a Kafka topic, only
ordering **within** a partition. This is the single most common surprise for people new to
Kafka, and it is a direct consequence of partitions being independent so they can live on
different machines. If you need strict ordering for a particular user, the answer is not
"turn off partitioning", it is "use that user as the key", which is exactly what we did.

### Parsing the JSON into real columns

Right now `json_value` is one long string. To treat `user` and `page` as columns, Spark has
to be told the structure, because unlike Python it will not guess at runtime.

**The new idea: a schema.** A **schema** is the list of column names and types. Spark is
strongly typed and insists on knowing the shape of your data before it reads it.

**Line by line.**

| Code | Meaning |
|---|---|
| `StructType([...])` | A record made of fields. This is how Spark describes any structured shape. |
| `StructField('user', StringType())` | One field: name `user`, type text. |
| `F.from_json('json_value', schema)` | Parse that string column into a structured column, using the schema. |
| `.select('event.user', ...)` | Reach inside the parsed structure with a dot, the same way you would in SQL or a Python object. |
| `.groupBy('user', 'partition').count()` | Group rows and count them, exactly like SQL's `GROUP BY`. |

**Why specify the schema instead of letting Spark infer it?** Inference means reading part of
the data first just to guess, which costs an extra pass and can guess differently tomorrow if
the sample changes. On a stream it is worse: there is no "all the data" to sample. Supplying
the schema is faster and predictable, and on streams it is usually required.

In [ ]:
# Now parse the JSON into real columns.
schema = StructType([
    StructField('user', StringType()),
    StructField('page', StringType()),
    StructField('ts',   StringType()),
])

parsed = (clicks
          .withColumn('event', F.from_json('json_value', schema))
          .select('user_key', 'partition',
                  'event.user', 'event.page', 'event.ts'))
parsed.show(truncate=False)

print('which partition did each user land in?')
parsed.groupBy('user', 'partition').count().orderBy('user').show()

+--------+---------+------+-----------+--------------------------+
|user_key|partition|user  |page       |ts                        |
+--------+---------+------+-----------+--------------------------+
|alice   |0        |alice |/checkout  |2026-10-14T17:26:08.792804|
|bob     |0        |bob   |/checkout  |2026-10-14T17:26:08.792818|
|alice   |0        |alice |/home      |2026-10-14T17:26:08.792821|
|bob     |0        |bob   |/product/42|2026-10-14T17:26:08.792830|
|alice   |0        |alice |/checkout  |2026-10-14T17:26:08.792833|
|carlos  |1        |carlos|/checkout  |2026-10-14T17:26:08.792824|
|diana   |2        |diana |/search    |2026-10-14T17:26:08.792827|
|diana   |2        |diana |/checkout  |2026-10-14T17:26:08.792836|
+--------+---------+------+-----------+--------------------------+

which partition did each user land in?
+------+---------+-----+
|  user|partition|count|
+------+---------+-----+
| alice|        0|    3|
|   bob|        0|    2|
|carlos|        1|    1|
| dian

**What you should see.** The parsed table, then a grouped count where **each user appears in
exactly one partition**: alice all in one, carlos in another, and so on.

### Why that matters more than it looks

That is the key doing its job, and it is the property that makes per-user work cheap. Every
event for alice is already on the same machine, so counting her clicks needs **no data
movement**.

Send the same events with no key and Kafka would spread them round robin across all three
partitions. The count would still be correct, but Spark would first have to send every
alice row to one place, a **shuffle** across the network. Shuffles are where most of the
time goes in most distributed jobs, which is why "choose a good key" is a design decision
worth thinking about rather than an afterthought.

## Step 5. Now read it as a stream

This is the whole point of the lab.

### The idea: an unbounded table

Spark's trick is to treat a stream as **a table that keeps growing**. New messages are new
rows appended to the bottom. Your query is written against that table exactly as if it were
finite, and Spark re-runs it as rows arrive.

So you do not write "poll Kafka every two seconds, track which offsets I have seen, update
my running totals". You write `groupBy('user').count()` and Spark maintains it.

### Spot the difference

Compare this with Step 4. The only change is `spark.read` becoming `spark.readStream`.
Same format, same options, same casting, same parsing. Streaming in Spark is not a different
data model, it is a different **schedule**.

In [ ]:
stream = (spark.readStream.format('kafka')
          .option('kafka.bootstrap.servers', BOOTSTRAP)
          .option('subscribe', 'clicks')
          .option('startingOffsets', 'earliest')
          .load()
          .selectExpr("CAST(key AS STRING) AS user_key",
                      "CAST(value AS STRING) AS json_value")
          .withColumn('event', F.from_json('json_value', schema))
          .select('event.user', 'event.page',
                  F.to_timestamp('event.ts').alias('event_time')))

print('is this a streaming DataFrame?', stream.isStreaming)

is this a streaming DataFrame? True


**What you should see.** `is this a streaming DataFrame? True`.

Note that nothing has run yet. Defining a streaming DataFrame starts no work at all; the work
begins when you start a **query** in the next cell.

### Starting a query

**The new ideas.**

A **sink** is where results go. A **trigger** is how often. An **output mode** is what to
send each time.

**Output modes**, which confuse everyone at first:

| Mode | Sends | Use when |
|---|---|---|
| `complete` | The entire result table, recomputed | Aggregations whose old answers change, like a running count |
| `append` | Only brand new rows | Results that never change once written |
| `update` | Only rows that changed since last time | Aggregations, when you only want the deltas |

We use `complete` because a count of alice's clicks changes every time alice clicks, so
"only new rows" would make no sense.

**Line by line.**

| Code | Meaning |
|---|---|
| `stream.groupBy('user').count()` | Identical to the batch version. That is the point. |
| `def show_batch(df, batch_id)` | Your own function, which Spark will call. It receives that batch's result as an **ordinary DataFrame** plus a batch number. |
| `df.orderBy(F.desc('count')).collect()` | Sort by count descending and pull the rows into Python. `collect()` brings data to your machine, which is fine for four rows and dangerous for four million. |
| `.foreachBatch(show_batch)` | The sink: "for each micro-batch, call my function". |
| `.trigger(processingTime='2 seconds')` | Run a batch every two seconds. |
| `.start()` | Begin. Returns immediately; the query runs in the background. |
| `time.sleep(6)` | Wait so you can see a few batches. |

### Micro-batches

Spark does not process messages one at a time. Every two seconds it takes everything that
arrived and processes it as a small batch. That is **micro-batching**, and it is why Spark
Structured Streaming has latency measured in seconds rather than microseconds. For nearly
every business question that is a fine trade for getting the same API as batch.

### Why `foreachBatch`, and a warning

`foreachBatch` is the most useful sink to know, because inside it you have a normal
DataFrame and can do anything: print it, write it to a database, call an API, send an alert.

Avoid the `memory` sink for this. In Spark 4 a complete-mode memory table can report a row
count and then render an empty table, which looks exactly like a bug in your code when it is
not.

In [ ]:
# Count events per user, continuously.
per_user = stream.groupBy('user').count()


def show_batch(df, batch_id):
    # Called once per micro-batch, with that batch's result as a normal DataFrame.
    rows = df.orderBy(F.desc('count')).collect()
    line = ', '.join('%s=%d' % (r['user'], r['count']) for r in rows if r['user'])
    print('   [batch %2d] %s' % (batch_id, line or '(nothing yet)'))


query = (per_user.writeStream
         .outputMode('complete')              # recompute the whole answer each time
         .foreachBatch(show_batch)            # and hand it to our function
         .trigger(processingTime='2 seconds')
         .start())

print('query started. Watch the batches appear below:')
time.sleep(6)

query started. Watch the batches appear below:
   [batch  0] alice=3, diana=2, bob=2, carlos=1


**What you should see.** One or two lines like
`[batch  0] alice=3, diana=2, bob=2, carlos=1`, matching the eight events from Step 3.

### Now send more and watch it react

The query above is **still running**. Send more events and the counts climb with no further
code from you.

**The new idea: a thread.** If we just looped and sent events, the cell would be busy
sending and you could not watch output appear. A **thread** runs code alongside the rest of
the program, so sending and watching happen at once.

| Code | Meaning |
|---|---|
| `import threading` | Python's threads. |
| `threading.Thread(target=drip, daemon=True)` | Make a thread that will run `drip`. `daemon=True` means "do not keep the program alive just for this", so a forgotten thread cannot hang the notebook. |
| `t.start()` | Begin it, and carry on immediately. |
| `time.sleep(14)` | The main code waits here while the thread sends in the background and the query keeps printing. |
| `query.stop()` | Stop this query before Step 6 starts another, so their output does not interleave. |

In [ ]:
import threading

def drip(n=20, delay=0.5):
    # Send n events, in the background, while the query keeps running.
    for _ in range(n):
        send([make_event()])
        time.sleep(delay)


print('sending 20 events. Watch the counts climb:')
t = threading.Thread(target=drip, daemon=True)
t.start()
time.sleep(14)          # long enough for about 7 more batches

query.stop()            # stop this one before Step 6 starts another
print()
print('done sending, and this query is stopped.')

sending 20 events. Watch the counts climb:
   [batch  1] alice=5, diana=3, bob=3, carlos=1
   [batch  2] bob=5, alice=5, diana=4, carlos=3
   [batch  3] diana=6, bob=6, alice=5, carlos=4
   [batch  4] diana=9, bob=6, carlos=5, alice=5
   [batch  5] diana=11, bob=7, carlos=5, alice=5

done sending, and this query is stopped.


**What you should see.** Around six batches with the numbers rising, something like:

```
   [batch  1] alice=5, diana=4, bob=2, carlos=1
   [batch  2] alice=8, diana=4, bob=3, carlos=1
   [batch  3] alice=10, diana=4, bob=3, carlos=3
```

### Stop and look at what just happened

The counts climbed on their own. You wrote no loop, no timer, no offset tracking, no
"where did I get to last time" bookkeeping.

That is the difference between batch and stream processing. You stopped describing **when**
to compute and described only **what** to compute. Spark handled the when, including
remembering its position in each partition so that nothing was counted twice.

## Step 6. Count events in time windows

"Clicks per user, ever" is rarely a useful question. "Clicks per user in the last 30
seconds" is. That needs two new ideas.

### Idea 1: event time versus processing time

| | Meaning |
|---|---|
| **Event time** | When the thing actually happened, carried in the data (our `ts` field) |
| **Processing time** | When Spark happened to look at it |

These differ, sometimes by a lot: a phone that was offline for an hour uploads events with
an hour old event time. Almost always you want event time, because "how many clicks
happened between 9:00 and 9:30" is a question about reality, not about your server's
schedule. That is why Step 5 converted `ts` with `F.to_timestamp`.

### Idea 2: the watermark

A **window** groups events into time buckets: ours are 30 seconds wide, so 09:35:30 to
09:36:00 is one bucket.

Here is the problem. If an event could arrive an hour late, Spark must keep every bucket
open forever just in case, and its memory grows without limit until the job dies.

A **watermark** is your promise about lateness. `withWatermark('event_time', '1 minute')`
says: "I accept events up to a minute late; anything later may be dropped." That lets Spark
finalize a bucket once the watermark passes it, and release the memory.

The trade is explicit and it is yours to make: a longer watermark catches more stragglers
and uses more memory. There is no setting that gives you both.

**Line by line.**

| Code | Meaning |
|---|---|
| `F.window('event_time', '30 seconds')` | Assign each row to a 30 second bucket. Produces a column with `start` and `end` inside it. |
| `.groupBy(F.window(...), 'user')` | Group by bucket **and** user, so you get a count per user per bucket. |
| `F.col('window.start')` | Reach inside the window column for its start time. |
| `r['start'].strftime('%H:%M:%S')` | Format a timestamp as hours:minutes:seconds for readable output. |

In [ ]:
windowed = (stream
            .withWatermark('event_time', '1 minute')
            .groupBy(F.window('event_time', '30 seconds'), 'user')
            .count())


def show_windows(df, batch_id):
    rows = (df.select(F.col('window.start').alias('start'), 'user', 'count')
              .orderBy('start', 'user').collect())
    print('   [batch %2d]' % batch_id)
    for r in rows:
        if r['user']:
            print('        %s  %-8s %d' % (r['start'].strftime('%H:%M:%S'),
                                           r['user'], r['count']))
    if not rows:
        print('        (nothing yet)')


wq = (windowed.writeStream
      .outputMode('complete')
      .foreachBatch(show_windows)
      .trigger(processingTime='4 seconds')
      .start())

print('windowed query started, sending 15 more events:')
t2 = threading.Thread(target=lambda: drip(15, 0.5), daemon=True)
t2.start()
time.sleep(16)
wq.stop()
print()
print('stopped.')

windowed query started, sending 15 more events:
   [batch  0]
        17:26:00  alice    3
        17:26:00  bob      2
        17:26:00  carlos   1
        17:26:00  diana    2
        17:27:00  alice    2
        17:27:00  bob      5
        17:27:00  carlos   4
        17:27:00  diana    10
   [batch  1]
        17:26:00  alice    3
        17:26:00  bob      2
        17:26:00  carlos   1
        17:26:00  diana    2
        17:27:00  alice    3
        17:27:00  bob      5
        17:27:00  carlos   5
        17:27:00  diana    10
        17:27:30  alice    1
        17:27:30  bob      2
        17:27:30  carlos   1
        17:27:30  diana    1


**What you should see.** Rows grouped into two or three buckets, like:

```
   [batch  2]
        09:35:30  alice    3
        09:35:30  bob      2
        09:36:00  alice    12
        09:36:00  bob      6
```

### The thing to actually notice

Watch the **older** bucket across successive batches. It stops changing, while the newest
one keeps growing. In the verified run, `09:35:30 alice` sat at 3 for every batch while
`09:36:00 alice` climbed 9, 12, 14.

That is the watermark working. Once the clock moved past the old bucket, Spark finalized it
and stopped spending memory on it. Nothing in your code said to do that; it followed from
one promise about lateness.

### Why this is the shape of nearly every real streaming job

"Errors per service per minute", "orders per region per hour", "failed logins per account
per five minutes" are all this exact query with different column names. Learn this one and
you have learned most of them.

**Check yourself.** You set the watermark to 1 minute. An event arrives with an event time of 3 minutes ago. What happens, and how would you keep it?

<details>
<summary>Click for the answer</summary>

It is dropped, because its bucket has already been finalized and released. To keep it you would widen the watermark, say `withWatermark('event_time', '5 minutes')`, which makes Spark hold every bucket open five minutes longer and therefore use more memory. That is the whole trade: completeness against memory. Decide it from how late your data realistically arrives, not by guessing.

</details>

## Step 7. Shut everything down

Colab throws the machine away anyway, so nothing here is strictly necessary. Do it anyway,
because on a real cluster the difference between a clean exit and a job that hangs holding
resources is exactly this cell.

**Order matters.** Stop the queries first, then the producer, then Spark, then the broker.
Stop the broker first and the still-running queries would spend their last seconds throwing
connection errors.

| Code | Meaning |
|---|---|
| `spark.streams.active` | Every query still running. Looping over it means you cannot forget one. |
| `q.stop()` | Stop a query cleanly, finishing its current batch rather than being killed mid-write. |
| `producer.close()` | Flush anything buffered and close the connection. |
| `broker.terminate()` | Ask the broker to shut down, the polite signal rather than a kill. |
| `broker.wait(timeout=30)` | Wait for it to actually exit, up to 30 seconds. Without this the next cell could run while the broker is still stopping. |

In [ ]:
for q in spark.streams.active:
    print('stopping query', q.id)
    q.stop()

producer.close()
spark.stop()
broker.terminate()
broker.wait(timeout=30)
print()
print('all stopped. Kafka log is at /content/broker.log if you want to read it.')

---

# What to take away

## The five facts

1. **A topic is a log you append to.** Partitions slice it so it can live on many machines.
2. **The key decides the partition**, so the same key always lands in the same place. That is
   what makes per-key work cheap and avoids a shuffle.
3. **Offsets are per partition.** There is no global order in a topic, only order within a
   partition.
4. **`read` and `readStream` give the same columns.** Streaming is not a different data
   model, it is a different schedule.
5. **A watermark is a promise about lateness**, and it is the only thing keeping windowed
   state from growing forever.

## The habits worth more than the facts

- **`flush()` before you read.** Most "my stream is missing data" problems are this.
- **Cast `key` and `value` to strings first.** Kafka always hands you bytes.
- **Give Spark a schema.** Do not make it guess, especially on a stream.
- **Detect, do not assume.** Step 1 checks which Kafka layout it got rather than hoping.
- **Measure before rewriting.** Which brings us to the next section.

## Why this lab is fast, which is itself the lesson

Version 1 of this notebook took over 15 minutes to set up. Version 2 takes about two.

| What changed | Before | After |
|---|---|---|
| Kafka from the CDN, not the throttled archive | about 17 min | about 10 s |
| Connector jars fetched directly, not resolved by Ivy | minutes, sometimes stalls | about 1.4 s |
| One long lived producer, not a JVM per message | 1 to 2 s per message | 0.115 s for 8 messages |

**Not one line of Spark or Kafka logic changed.** What changed was where the bytes came from
and how often a process was started.

That is where almost all the time goes in real data engineering too. Not in the computation,
but in moving data and in paying startup costs over and over. When a job of yours is slow,
measure first, and check the boring things before you rewrite the clever ones.

---

# Glossary

Everything this notebook used, in one place.

## Kafka

| Term | Meaning |
|---|---|
| **Broker** | The Kafka server. Stores messages and serves them. |
| **Topic** | A named, append only log of messages. |
| **Partition** | One slice of a topic. The unit of parallelism and of ordering. |
| **Offset** | A message's position within one partition, counting from 0. |
| **Key** | Optional value attached to a message; its hash picks the partition. |
| **Producer** | A program that writes to a topic. |
| **Consumer** | A program that reads from a topic. |
| **Bootstrap server** | The address a client contacts first to discover the cluster. |
| **Replication factor** | How many copies of each partition exist. 1 here, 3 in production. |
| **ISR** | In Sync Replicas: the copies that are currently up to date. |
| **KRaft** | Modern Kafka's built in coordination, replacing the old ZooKeeper requirement. |
| **flush** | Block until the broker has acknowledged every buffered message. |

## Spark

| Term | Meaning |
|---|---|
| **SparkSession** | Your handle on the engine. Everything starts here. |
| **DataFrame** | A table with named, typed columns. |
| **Schema** | The list of column names and types. |
| **Lazy evaluation** | Spark plans but does nothing until an action demands a result. |
| **Action** | An operation that forces real work, such as `count()`, `show()`, `collect()`. |
| **Shuffle** | Moving data across the network to regroup it. Usually the expensive part. |
| **readStream** | The streaming counterpart of `read`. |
| **Micro-batch** | The small chunk Spark processes per trigger. |
| **Trigger** | How often to run a micro-batch. |
| **Output mode** | What to emit each batch: `complete`, `append` or `update`. |
| **Sink** | Where results go. `foreachBatch` hands each batch to your own function. |
| **Event time** | When something happened, per the data. |
| **Processing time** | When Spark looked at it. |
| **Window** | A time bucket events are grouped into. |
| **Watermark** | Your promise about how late data may be, which lets Spark release state. |

## Everything else

| Term | Meaning |
|---|---|
| **JVM** | Java Virtual Machine. Kafka and Spark both run on it; your Python only drives them. |
| **JDK / JRE** | The Java kit with a compiler, versus the runtime that only runs programs. |
| **jar** | A zip of compiled Java code. Java's version of a package. |
| **Maven** | The main public repository of Java libraries. |
| **Ivy** | The dependency resolver Spark uses to fetch from Maven. Slow in Colab. |
| **JSON** | A text format for structured data; looks much like a Python dict. |
| **Serialization** | Turning an object into bytes. Deserialization is the reverse. |
| **UTF-8** | The standard way of encoding text as bytes. |
| **Environment variable** | A setting, such as `JAVA_HOME`, readable by any program started from here. |
| **PATH** | The folders the shell searches for commands. |
| **Background process** | A program started without waiting for it to finish. |
| **Thread** | Code running alongside the rest of your program. |
| **lambda** | A one line unnamed function. |
| **Pinning** | Fixing a dependency to an exact version, like `pyspark==4.0.1`. |

---

# If something goes wrong

| Symptom | What it means | Fix |
|---|---|---|
| Broker did not start | Usually the runtime was recycled and `/content` was wiped | `!tail -40 /content/broker.log`, then run from Step 0 |
| `Failed to find data source: kafka` | The jars cell did not finish | Re-run it, then **Runtime > Restart session**, then run from Step 0 |
| A query prints `(nothing yet)` forever | No data, or reading from the wrong place | Check you called `producer.flush()`, check the topic name, and check `startingOffsets` is `earliest` not `latest` |
| `NameError: name 'spark' is not defined` | A cell was run out of order | Run from the top. Every cell depends on the ones above it. |
| Download is slow again | The CDN dropped that version | Nothing to do: the cell falls back to the archive automatically. Slow but it works. |
| Counts look too high | You ran Step 3 more than once | Each run adds more events. Restart the runtime for a clean slate. |
| `Port 9092 already in use` | A broker from an earlier run is still up | Restart the runtime, or `!pkill -f kafka-server-start` |

---

# Where to go next

- Change the topic to **1 partition** and re-run Step 4. Where does everyone land, and what
  does that do to the parallelism?
- Remove the `key=` from `producer.send` and re-run. Watch the per-user-per-partition table
  from Step 4 change, and think about what that costs Spark.
- Change the window from 30 seconds to 10 and the watermark from 1 minute to 20 seconds. How
  does the output change?
- Try `outputMode('update')` instead of `complete` in Step 5 and compare what gets printed.